# Turkish Gemma T1 Server (Windows)
Start a FastAPI proxy over the local Ollama model and expose it through ngrok.


In [3]:
# Set local server configuration.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
OLLAMA_MODEL = "turkish-gemma-t1-q4km"
OLLAMA_URL = "http://127.0.0.1:11434/api/generate"
API_HOST = "127.0.0.1"
API_PORT = 8000
API_KEY = "alfabeta11!"
NUM_CTX = 4096
NUM_PREDICT = 1024
SYSTEM_MESSAGE = ""
CHAT_CLIENT_PATH = PROJECT_ROOT / "chat_client.html"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("OLLAMA_MODEL =", OLLAMA_MODEL)


PROJECT_ROOT = g:\Drive'ım\training-embedding
OLLAMA_MODEL = turkish-gemma-t1-q4km


In [4]:
# Define the FastAPI proxy app.
from html import escape
from typing import Optional
import re

import httpx
from fastapi import FastAPI, Header, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field

SESSION_STORE: dict[str, list[dict[str, str]]] = {}
THINK_RE = re.compile(r"<think>.*?</think>", flags=re.DOTALL | re.IGNORECASE)

app = FastAPI(title="Ollama Proxy")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)


class Turn(BaseModel):
    role: str
    content: str


class GenerateRequest(BaseModel):
    user_message: str


class ChatRequest(BaseModel):
    user_message: str
    session_id: Optional[str] = None
    history: list[Turn] = Field(default_factory=list)


class GenerateResponse(BaseModel):
    text: str
    session_id: Optional[str] = None


def _require_api_key(x_api_key: Optional[str]) -> None:
    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="Unauthorized")


def _clean_text(text: str) -> str:
    cleaned = THINK_RE.sub("", text or "")
    cleaned = cleaned.replace("<start_of_turn>model", "")
    cleaned = cleaned.replace("<end_of_turn>", "")
    if "<start_of_turn>user" in cleaned:
        cleaned = cleaned.split("<start_of_turn>user", 1)[0]
    if "<think>" in cleaned:
        cleaned = cleaned.split("<think>", 1)[0]
    return cleaned.strip()


def _build_prompt(user_message: str, history: list[Turn]) -> str:
    chunks: list[str] = []
    if SYSTEM_MESSAGE:
        chunks.append(f"<start_of_turn>user\\n{SYSTEM_MESSAGE}\\n<end_of_turn>\\n")
        chunks.append("<start_of_turn>model\\nTamam.\\n<end_of_turn>\\n")
    for turn in history:
        role = turn.role.strip().lower()
        content = turn.content.strip()
        if role == "assistant":
            chunks.append(f"<start_of_turn>model\\n{content}\\n<end_of_turn>\\n")
        else:
            chunks.append(f"<start_of_turn>user\\n{content}\\n<end_of_turn>\\n")
    chunks.append(f"<start_of_turn>user\\n{user_message}\\n<end_of_turn>\\n<start_of_turn>model\\n")
    return "".join(chunks)


def _chat_html(api_base: str) -> str:
    chat_html = CHAT_CLIENT_PATH.read_text(encoding="utf-8")
    config_script = (
        "<script>"
        f"window.GEMMA_API_BASE = \"{escape(api_base, quote=True)}\";"
        f"window.GEMMA_API_KEY = \"{escape(API_KEY, quote=True)}\";"
        "</script>"
    )
    return chat_html.replace("</head>", config_script + "\n</head>", 1)


async def _call_ollama(prompt: str) -> str:
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {
            "num_ctx": NUM_CTX,
            "num_predict": NUM_PREDICT,
            "stop": ["<end_of_turn>", "<start_of_turn>user"],
        },
    }
    async with httpx.AsyncClient(timeout=240) as client:
        resp = await client.post(OLLAMA_URL, json=payload)
    resp.raise_for_status()
    return _clean_text((resp.json().get("response") or "").strip())


@app.get("/", response_class=HTMLResponse)
async def home() -> str:
    return (
        "<!doctype html><html><body style='font-family:sans-serif;padding:24px'>"
        "<h2>Turkish Gemma T1</h2>"
        "<p>Model server is up.</p>"
        "<a href='/app'>Open Chat</a>"
        "</body></html>"
    )


@app.get("/app", response_class=HTMLResponse)
async def chat_app(request: Request) -> str:
    api_base = f"{request.url.scheme}://{request.headers['host']}"
    return _chat_html(api_base)


@app.post("/generate", response_model=GenerateResponse)
async def generate(req: GenerateRequest, x_api_key: Optional[str] = Header(default=None, alias="x-api-key")):
    _require_api_key(x_api_key)
    text = await _call_ollama(_build_prompt(req.user_message, []))
    return GenerateResponse(text=text)


@app.post("/chat", response_model=GenerateResponse)
async def chat(req: ChatRequest, x_api_key: Optional[str] = Header(default=None, alias="x-api-key")):
    _require_api_key(x_api_key)
    session_id = (req.session_id or "default").strip()
    history = req.history or [Turn(**turn) for turn in SESSION_STORE.get(session_id, [])]
    text = await _call_ollama(_build_prompt(req.user_message, history))
    updated = [*history, Turn(role="user", content=req.user_message), Turn(role="assistant", content=text)]
    SESSION_STORE[session_id] = [turn.model_dump() for turn in updated[-24:]]
    return GenerateResponse(text=text, session_id=session_id)


In [ ]:
# Start the FastAPI server in a background thread.
import threading

import uvicorn

server = uvicorn.Server(
    uvicorn.Config(app, host=API_HOST, port=API_PORT, log_level="info")
)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
print(f"Local app: http://{API_HOST}:{API_PORT}/app")


Local app: http://127.0.0.1:8000/app


INFO:     Started server process [17164]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


In [6]:
# Start ngrok and print the public URL.
from pyngrok import ngrok

ngrok.set_auth_token(__import__("os").environ.get("NGROK_AUTH_TOKEN") or __import__("pathlib").Path(".env").read_text(encoding="utf-8").split("NGROK_AUTH_TOKEN=", 1)[1].splitlines()[0].strip())
ngrok.kill()
public_url = ngrok.connect(API_PORT, bind_tls=True).public_url.rstrip("/")
print("Public app:", public_url + "/app")
print("Public API:", public_url)


Public app: https://tensive-nonconceptually-alease.ngrok-free.dev/app
Public API: https://tensive-nonconceptually-alease.ngrok-free.dev


In [ ]:
# Smoke test the local proxy.
import requests

resp = requests.post(
    f"http://{API_HOST}:{API_PORT}/generate",
    headers={"x-api-key": API_KEY},
    json={"user_message": "Merhaba! Kısaca kendini tanıtır mısın?"},
    timeout=600,
)
resp.raise_for_status()
print(resp.json()["text"])


INFO:     78.170.154.226:0 - "GET / HTTP/1.1" 200 OK
INFO:     78.170.154.226:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:53828 - "POST /generate HTTP/1.1" 200 OK
Merhaba! 😊 Ben DeepSeek-R1 olarak adlandırılan bir yapay zeka asistanıyım. Metin tabanlı sohbetlerle destek olmak için tasarlandım — sorularını yanıtlayabilir, bilgi paylaşabilirim, metin yazmanıza yardımcı olabilirim (makaleler, özetler, kodlar vb.), dil çevirileri yapabilirim ve daha birçok konuda fikir verebilirim.  

Kısacası: **İnsan gibi düşünemem ama öğrendiğim bilgilerle sana mantıklı yanıtlar üretmeye çalışırım.** Amacım karmaşık soruları basitleştirmek, öğrenmeni kolaylaştırmak veya işlerini hızlandırmak! 🚀  

Şimdi sıra sende — bana ne danışmak istersin? 😊


INFO:     78.170.154.226:0 - "GET /app HTTP/1.1" 200 OK
INFO:     78.170.154.226:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     78.170.154.226:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     78.170.154.226:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     78.170.154.226:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     78.170.154.226:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     78.170.154.226:0 - "GET / HTTP/1.1" 200 OK
INFO:     78.170.154.226:0 - "GET / HTTP/1.1" 200 OK
INFO:     78.170.154.226:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     78.170.154.226:0 - "GET /app HTTP/1.1" 200 OK
INFO:     78.170.154.226:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     78.170.154.226:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     78.170.154.226:0 - "POST /chat HTTP/1.1" 200 OK
INFO:     78.170.154.226:0 - "GET /apple-touch-icon-precomposed.png HTTP/1.1" 404 Not Found
INFO:     78.170.154.226:0 - "GET /apple-touch-icon.png HTTP/1.1" 404 Not Found
INFO:     78.170.154.226:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO